In [2]:
import os
import time
import pandas as pd
import urllib
from sqlalchemy import create_engine
from sqlalchemy.types import String
import logging



logging.basicConfig(
    filename="logs/ingestion_db.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)



params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=KHUSHI-GUPTA\\SQLEXPRESS;"
    "DATABASE=Vendor_Management_System;"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    fast_executemany=True
)

# ------------------- INGEST FUNCTION -------------------
def ingest_db(df, table_name, engine, mode="replace"):
    dtype_map = {col: String(255) for col in df.columns}

    df.to_sql(
        table_name,
        con=engine,
        if_exists=mode,     # append or replace
        index=False,
        chunksize=1000,
        dtype=dtype_map
    )

# ------------------- READ & LOAD CSVs as dataframe and ingest into db -------------------
def load_raw_data():
    start=time.time()
    for file in os.listdir("data"):
        if file.endswith(".csv"):
            df = pd.read_csv(os.path.join("data", file))
            logging.info(f'Ingesting {file} in db')
            ingest_db(df, file[:-4], engine, mode="replace")
    end=time.time()
    total_time=(end-start)/60
    logging.info('------------Ingestion Complete------------')
    logging.info(f'Total time take is {total_time} minutes')
    
    
if __name__=='__main__':
    load_raw_data()

In [1]:
# import logging
# import os

# if not os.path.exists("logs"):
#     os.makedirs("logs")

# logging.basicConfig(
#     filename="logs/ingestion_db.log",
#     level=logging.DEBUG,
#     format="%(asctime)s - %(levelname)s - %(message)s",
#     filemode="a"
# )
